In [0]:
# ==========================================
# 1. INITIAL CONFIGURATIONS
# ==========================================
#Replace the below credentials with your own credentials (created using the Automated Cloud Setup).
# Defines the service principal credentials and ADLS Gen2 pathing 
# Required for Auto Loader (CloudFiles) and Delta Lake operations.
tenant_id="2ae11cb7-4c20-4921-8f30-a2c46145abb9"
subscription_id="9dee02a6-39b8-43fe-821e-0af9438ce3dc"
resource_group_name='raji_resource_group'
sp_client_id="a14e63cc-166b-42c7-8f3e-fb73e2f81f75"
sp_secret_scope_name='rajeswari-scope'
sp_secret_value="sp-secret"
storage_account = "rajeswariadlsacct1.dfs.core.windows.net"
container_name="rajeswaricontainer1"
folder_name='landing-zone'

terminate_seconds=200


In [0]:
# ==========================================
# 1. PATHS & TABLE CONFIGURATIONS
# ==========================================
# ==========================================
# SOURCE PATHS
# ==========================================

atm_source_path = (
    "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/"
    "landing-zone/ATM Transactions /"
)


credit_source_path = (
    "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/"
    "landing-zone/Credit Card Stream/"
)

client_secret_value = dbutils.secrets.get(scope=sp_secret_scope_name, key="sp-secret")

# Table Names
bronze_table = "banking_bronze.transactions_bronze_tbl"
silver_table = "banking_silver.transactions_silver_tbl"
gold_fact_table = "banking_gold.fact_wide_transactions"
gold_aggr_table = "banking_gold.fact_account_transactions"
dim_account_scd1 = "banking_gold.dim_account_gold_scd1"#Ensure the tablename used here is same as the one used in the DLT pipeline code
dim_customer_scd1 = "banking_gold.dim_customer_gold_scd1"
dim_account_scd2 = "banking_gold.dim_account_gold_scd2"#Ensure the tablename used here is same as the one used in the DLT pipeline code
dim_customer_scd2 = "banking_gold.dim_customer_gold_scd2"
bronzeschemadir='bronzesch3'
bronzecheckpointdir="bronzeckpt15"
silvercheckpointdir="silverckpt15"
goldcheckpointdir1="gold1ckpt15"
goldcheckpointdir2="gold2ckpt15"

In [0]:

# ==========================================
# SPARK AUTH CONFIGURATION
# ==========================================

#spark.conf.set(f"fs.azure.account.fixed.vnet.lookup.{storage_account}", "true")
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}",
    sp_client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}",
    client_secret_value
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

# ==========================================
# AUTO LOADER COMMON CONFIG
# ==========================================

cloudFilesConf = {
    "cloudFiles.format": "csv",
    "header": "true",
    "cloudFiles.inferColumnTypes": "true",
    "cloudFiles.schemaEvolutionMode": "addNewColumns",
    "cloudFiles.useNotifications": "false",
    "cloudFiles.subscriptionId": subscription_id,
    "cloudFiles.resourceGroup": resource_group_name,
    "cloudFiles.tenantId": tenant_id,
    "cloudFiles.clientId": sp_client_id,
    "cloudFiles.clientSecret": client_secret_value,
    "cloudFiles.maxFilesPerTrigger": "10"
}

print("Spark Authentication Configured")
print("Auto Loader Configuration Ready")

In [0]:
from pyspark.sql.functions import *

# ==========================================
# PATHS
# ==========================================

atm_source_path = (
    "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/"
    "landing-zone/ATM Transactions /"
)


credit_source_path = (
    "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/"
    "landing-zone/Credit Card Stream/"
)

bronze_checkpoint = (
    "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/"
    "_checkpoints/new_bronze_atm_credit/"
)

silver_checkpoint = (
    "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/"
    "_checkpoints/new_silver_atm_credit/"
)

gold_fact_checkpoint = (
"abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/"
"_checkpoints/new_gold_fact_credit_v2/"
)

gold_aggr_checkpoint = (
    "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/"
    "_checkpoints/new_gold_aggr_credit/"
)


bronze_table = "telecom_rajeswari_workspace.banking_bronze.transactions_bronze_tbl"

silver_table = "telecom_rajeswari_workspace.banking_silver.transactions_silver_tbl"

gold_fact_table = "telecom_rajeswari_workspace.banking_gold.fact_wide_transactions"

gold_aggr_table = "telecom_rajeswari_workspace.banking_gold.fact_account_transactions"



# ==========================================
# BRONZE
# ATM + CREDIT CARD
# ==========================================


atm_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("cloudFiles.useNotifications","false")
    .option(
        "cloudFiles.schemaLocation",
        "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/_schema/new_atm/"
    )
    .option("header","true")
    .option("delimiter",",")
    .option("cloudFiles.inferColumnTypes","true")
    .load(atm_source_path)
)



credit_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("cloudFiles.useNotifications","false")
    .option(
        "cloudFiles.schemaLocation",
        "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/_schema/new_credit_card/"
    )
    .option("header","true")
    .option("delimiter",",")
    .option("cloudFiles.inferColumnTypes","true")
    .load(credit_source_path)
)



# Credit Card column mapping

credit_df = credit_df.select(
    "txn_id",
    "account_id",
    "card_id",
    "amount",
    "merchant_cat",
    "timestamp",
    "status"
)



# ATM + Credit Card combine

df_landing = (
    atm_df
    .unionByName(
        credit_df,
        allowMissingColumns=True
    )
)

bronze_df = (
    df_landing
    .withColumn(
        "ingestion_time",
        current_timestamp()
    )
)


query_bronze = (
    bronze_df.writeStream
    .queryName("Bronze_Ingestion_New")
    .format("delta")
    .option("checkpointLocation",bronze_checkpoint)
    .outputMode("append")
    .option("mergeSchema","true")
    .toTable(bronze_table)
)


print("Bronze Started")

# ==========================================
# SILVER LAYER
# ==========================================

bronze_stream = (
    spark.readStream
    .table(
        "telecom_rajeswari_workspace.banking_bronze.transactions_bronze_tbl"
    )
)


silver_df = (
    bronze_stream

    .withColumnRenamed(
        "account_id",
        "ACCOUNT_ID"
    )

    .withColumn(
        "transaction_timestamp",
        to_timestamp(col("timestamp"))
    )

    .drop("timestamp")

    .filter(
        col("transaction_timestamp").isNotNull()
    )

    .withWatermark(
        "transaction_timestamp",
        "10 minutes"
    )

    .dropDuplicates(["txn_id"])

    .withColumn(
        "transaction_date",
        to_date("transaction_timestamp")
    )

    .withColumn(
        "transaction_hour",
        hour("transaction_timestamp")
    )

    .withColumn(
        "transaction_status",
        when(
            col("status")=="SUCCESS",
            "SUCCESS"
        )
        .otherwise("FAILED")
    )

    .withColumn(
        "amount_bucket",
        when(col("amount") < 1000,"LOW")
        .when(col("amount") < 10000,"MEDIUM")
        .otherwise("HIGH")
    )

    .withColumn(
        "silver_load_time",
        current_timestamp()
    )
)



query_silver = (
    silver_df.writeStream
    .queryName("Silver_Ingestion_New")
    .format("delta")
    .option(
        "checkpointLocation",
        "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/_checkpoints/new_silver_atm_credit/"
    )
    .outputMode("append")
    .option("mergeSchema","true")
    .toTable(
        "telecom_rajeswari_workspace.banking_silver.transactions_silver_tbl"
    )
)


print("Silver Started")

# ==========================================
# 4. GOLD LAYER (Enrichment & Aggregation)
# ==========================================

from pyspark.sql.functions import *
silver_table = "telecom_rajeswari_workspace.banking_silver.transactions_silver_tbl"

# Read Silver Streaming Table
silver_df_stream = (
    spark.readStream
    .table(silver_table)
)


# ==========================================
# 4a. GOLD FACT TABLE
# Silver Transaction + Gold Dimensions
# ==========================================


dim_account_table = (
    "telecom_rajeswari_workspace.banking_gold.dim_account_gold_scd1"
)

dim_customer_table = (
    "telecom_rajeswari_workspace.banking_gold.dim_customer_gold_scd1"
)


# Static Dimension Tables

dim_account = (
    spark.table(dim_account_table)
    .select(
        "ACCOUNT_ID",
        "CUST_ID",
        "TYPE",
        "BALANCE",
        "STATUS"
    )
)


dim_customer = (
    spark.table(dim_customer_table)
    .select(
        "CUST_ID",
        "NAME_MASKED",
        "Risk_Score",
        "KYC_Status"
    )
)



# Streaming + Static Join

gold_fact_enriched = (

    silver_df_stream.alias("f")

    .join(
        dim_account.alias("a"),
        col("f.ACCOUNT_ID") == col("a.ACCOUNT_ID"),
        "left"
    )

    .join(
        dim_customer.alias("c"),
        col("a.CUST_ID") == col("c.CUST_ID"),
        "left"
    )

    .select(

        col("f.txn_id"),
        col("f.ACCOUNT_ID"),
        col("f.transaction_timestamp"),
        col("f.amount"),
        col("f.card_id"),
        col("f.merchant_cat"),
        col("f.transaction_date"),
        col("f.transaction_hour"),
        col("f.transaction_status"),
        col("f.amount_bucket"),
        col("f.silver_load_time"),

        col("a.CUST_ID"),
        col("c.NAME_MASKED"),
        col("c.Risk_Score"),
        col("c.KYC_Status"),

        col("a.TYPE"),
        col("a.BALANCE"),
        col("a.STATUS"),

        current_timestamp().alias(
            "gold_load_time"
        )

    )

    # Gold level duplicate protection
    .dropDuplicates(
    [
        "txn_id",
        "transaction_timestamp"
    ]
)
)
# ==========================================
# START GOLD FACT STREAM
# ==========================================

query_gold_fact = (

    gold_fact_enriched

    .writeStream

    .queryName(
        "Gold_Fact_Wide_Ingestion"
    )

    .format("delta")

    .option(
        "checkpointLocation",
        gold_fact_checkpoint
    )

    .outputMode(
        "append"
    )

    .option(
        "mergeSchema",
        "true"
    )

    .toTable(
        gold_fact_table
    )

)


print("Gold fact_wide_transactions Started")
# ==========================================
# 4b. GOLD AGGREGATION TABLE
# ==========================================


gold_aggr_df = (

    silver_df_stream

    .withWatermark(
        "transaction_timestamp",
        "10 minutes"
    )

    .groupBy(
        "transaction_date",
        "ACCOUNT_ID"
    )

    .agg(

        count("*")
        .alias("total_transactions"),


        sum("amount")
        .alias("total_amount"),


        avg("amount")
        .alias("average_amount"),


        sum(
            when(
                col("transaction_status") == "SUCCESS",
                1
            )
            .otherwise(0)
        )
        .alias(
            "successful_transactions"
        )

    )

)



query_gold_aggr = (

    gold_aggr_df

    .writeStream

    .queryName(
        "Gold_Aggregation_Ingestion_New"
    )

    .format("delta")

    .option(
        "checkpointLocation",
        "abfss://rajeswaricontainer1@rajeswariadlsacct1.dfs.core.windows.net/_checkpoints/new_gold_aggr_credit/"
    )

    .outputMode(
        "complete"
    )

    .option(
        "mergeSchema",
        "true"
    )

    .toTable(
        gold_aggr_table
    )

)



# ==========================================
# STREAM STATUS
# ==========================================


active_queries = [
    query_bronze,
    query_silver,
    query_gold_fact,
    query_gold_aggr
]


print("Gold Streams Started")


for q in spark.streams.active:
    print(
        q.name,
        q.isActive
    )

